In [1]:
# General Imports
import sys
import datetime
import logging
# Math Imports
import numpy
import pandas
import matplotlib.pyplot as plt
import torch
import lightning
# Local imports
from cocodeel.dataset import CovarDataset
from cocodeel.model import NeuralNetwork
from cocodeel.model import CovarNeuralNetwork

In [20]:
DEVICE = "cuda:0"
#torch.set_default_device(DEVICE)
torch.set_default_dtype(torch.float)
torch.manual_seed(0)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [33]:
# Data generation.
def simulate_data(N=3000, q=256, p=2, confounding = True):
    # Random covariates.
    z0 = torch.ones(p)
    Z = z0 + torch.randn(N, p)
    Z = torch.cat((torch.ones(N, 1), Z), 1)
    # Confounded random network inputs.
    dimX = 256 * 256
    delta = torch.zeros(p, dimX)
    if confounding:
        delta[:, dimX//2:] = torch.ones(p, dimX//2)
    X = Z[:,1:] @ delta + torch.randn(N, dimX)
    with torch.no_grad():
        feature_extractor = torch.nn.Sequential(
            torch.nn.Linear(dimX, q),
            torch.nn.ReLU())
        phiX = feature_extractor(X)
    #delta = torch.zeros(p, q)
    #if confounding:
    #    delta[:, q//2:] = torch.ones(p, q//2) * 4 / (p**.5 * q**.5)
    #phiX = Z[:,1:] @ delta + torch.randn(N, q)
    # True effects.
    fx = phiX @ torch.ones(q) / q**.5
    fz = Z[:,1:] @ torch.ones(p) / p**.5
    # Output.
    y = fx + fz + torch.randn(N)
    return phiX, Z, y, fx, fz

def create_dataloaders(X, Z, y, batch_size=256):
    N = len(y)
    # Create training, validation and test data.
    train_data = CovarDataset(X[:N // 3], Z[:N // 3], y[:N // 3])
    val_data = CovarDataset(X[N // 3:2 * N // 3], Z[N // 3:2 * N // 3], y[N // 3:2 * N // 3])
    test_data = CovarDataset(X[2 * N // 3:], Z[2 * N // 3:], y[2 * N // 3:])
    # Create dataloaders.
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [19]:
# Model Backbones.
class IdentityBackbone(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.model = torch.nn.Identity()
    def forward(self, x):
        return self.model(x)

class ShallowBackbone(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.model = torch.nn.Sequential(
            torch.nn.Linear(in_features, out_features),
            torch.nn.ReLU()
        )
    def forward(self, x):
        return self.model(x)

def estimate_models(train_loader, val_loader, params):
    # Estimate Baseline Neural Network
    net = NeuralNetwork(**params).to(DEVICE)
    model_checkpoint = lightning.pytorch.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
    early_stop_callback = lightning.pytorch.callbacks.EarlyStopping(monitor="val_loss", patience=64, mode="min")
    trainer = lightning.Trainer(max_epochs=1000, enable_progress_bar=False, callbacks=[model_checkpoint, early_stop_callback])
    trainer.fit(net, train_loader, val_loader)
    net = NeuralNetwork.load_from_checkpoint(model_checkpoint.best_model_path).to(DEVICE)

    # Estimate Baseline Neural Network with Covariates
    net_covar = CovarNeuralNetwork(**params).to(DEVICE)
    model_checkpoint = lightning.pytorch.callbacks.ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
    early_stop_callback = lightning.pytorch.callbacks.EarlyStopping(monitor="val_loss", patience=64, mode="min")
    trainer = lightning.Trainer(max_epochs=1000, enable_progress_bar=False, callbacks=[model_checkpoint, early_stop_callback])
    trainer.fit(net_covar, train_loader, val_loader)
    net_covar = CovarNeuralNetwork.load_from_checkpoint(model_checkpoint.best_model_path).to(DEVICE)

    return {"NN": net, "Covar NN": net_covar}

In [9]:
# Mode Evaluation.
def get_residuals(model, test_loader):
    model.eval()
    model.to('cpu')
    q, p = model.backbone.in_features, model.num_covars - 1
    yhat, fxhat, fzhat = [], [], []
    for batch in test_loader:
        X, Z, y = batch["image"], batch["covar"], batch["label"]
        # True effects.
        fx = X @ torch.ones(q) / q**.5
        fz = Z[:,1:] @ torch.ones(p) / p**.5
        with torch.no_grad():
            yhat.append(model.predict_step(batch, None).squeeze() - y)
            fxhat.append(model.predict_deep(batch).squeeze() - fx)
            fzhat.append(model.predict_struct(batch).squeeze() - fz)
    return {"yhat" : torch.cat(yhat), "fxhat" : torch.cat(fxhat), "fzhat" : torch.cat(fzhat)}

def mse_decomposition(rhat):
    return {"mse": torch.mean(rhat**2).numpy(), "bias": torch.mean(rhat).numpy(), "variance": torch.var(rhat).numpy()}


In [43]:
# Simulate data
N, q, p = 30000, 1024, 8
X, Z, y, fx, fz = simulate_data(N=N, q=q, p=p, confounding=False)
train_loader, val_loader, test_loader = create_dataloaders(X, Z, y, batch_size=256)

# Model Parameters
params = {
    "backbone": IdentityBackbone,
    "output_func": torch.nn.Identity,
    "loss_func": torch.nn.MSELoss,
    "optimizer": torch.optim.AdamW,
    "num_covars": p+1,
    "num_features": q,
    "backbone_params": {"in_features": q, "out_features": q},
    "optimizer_params": {"lr": 0.1, "weight_decay": 0.0001},
    "scheduler": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 16, "gamma": 0.5}
}

models = estimate_models(train_loader, val_loader, params)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name           | Type             | Params
----------------------------------------------------
0 | backbone       | IdentityBackbone | 0     
1 | deep_predictor | Linear           | 1.0 K 
2 | output_func    | Identity         | 0     
3 | loss_func      | MSELoss          | 0     
----------------------------------------------------
1.0 K     Trainable params
0         Non-trainable params
1.0 K     Total params
0.004     Total estimated model params size (MB)
Trainer will use only 1 of 2 GPUs because it is runn

In [44]:
print("Baseline NN")
r = get_residuals(models["NN"], test_loader)
print(mse_decomposition(r['yhat']))
print(mse_decomposition(r['fxhat']))
print(mse_decomposition(r['fzhat']))
print("Covariate NN")
r = get_residuals(models["Covar NN"], test_loader)
print(mse_decomposition(r['yhat']))
print(mse_decomposition(r['fxhat']))
print(mse_decomposition(r['fzhat']))

Baseline NN
{'mse': array(2.1964982, dtype=float32), 'bias': array(0.09959641, dtype=float32), 'variance': array(2.1867974, dtype=float32)}
{'mse': array(8.279904, dtype=float32), 'bias': array(2.8454635, dtype=float32), 'variance': array(0.18326105, dtype=float32)}
{'mse': array(8.586173, dtype=float32), 'bias': array(-2.7511325, dtype=float32), 'variance': array(1.0175459, dtype=float32)}
Covariate NN
{'mse': array(1.123421, dtype=float32), 'bias': array(-0.10672221, dtype=float32), 'variance': array(1.1121427, dtype=float32)}
{'mse': array(0.10213594, dtype=float32), 'bias': array(-0.00822871, dtype=float32), 'variance': array(0.10207843, dtype=float32)}
{'mse': array(0.0128558, dtype=float32), 'bias': array(-0.1037588, dtype=float32), 'variance': array(0.00209012, dtype=float32)}


In [ ]:
# Simulation run over nsim iterations for different values of q
N, q, p = 30000, 128, 8
nsim = 1
results = []
for conf in [False, True]:
    for q in [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]:
        params_q = params.copy()
        params_q["num_covars"] = p+1
        params_q["num_features"] = q
        params_q["backbone_params"]["in_features"] = q
        params_q["backbone_params"]["out_features"] = q
        for i in range(nsim):
            X, Z, y, fx, fz = simulate_data(N=N, q=q, p=p, confounding=conf)
            train_loader, val_loader, test_loader = create_dataloaders(X, Z, y, batch_size=256)
            models = estimate_models(train_loader, val_loader, params_q)
            for key, model in models.items():
                residuals = get_residuals(model, test_loader)
                for key2, residual in residuals.items():
                    decomp = mse_decomposition(residual)
                    results.append({"confounding": conf, "q": q, "Run": i, "Model": key, "Metric": key2,
                                    "MSE": decomp['mse'], "Bias": decomp['bias'], "Var": decomp['variance']})
results = pandas.DataFrame.from_records(results)
if nsim > 5:
    results.to_csv(f"results_concurvity_{nsim}_{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}.csv")

In [253]:
results.loc[results.Model == "NN"][results.Metric == ("yhat")]

C:\Users\Manuel Pfeuffer\AppData\Local\Temp\ipykernel_11864\904893179.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  results.loc[results.Model == "NN"][results.Metric == ("yhat")]


,q,Run,Model,Metric,MSE,Bias,Var
0,16,0,NN,yhat,0.9847218,0.028865322,0.9848735
6,32,0,NN,yhat,1.0439142,-0.037447378,1.0435555
12,64,0,NN,yhat,1.0508076,0.11176783,1.039355
18,128,0,NN,yhat,1.1450481,0.057503816,1.1428843
24,256,0,NN,yhat,1.0868127,0.06623222,1.0835096
30,512,0,NN,yhat,1.0812067,-0.0428947,1.0804472
36,1024,0,NN,yhat,1.0751947,-0.025859272,1.0756016
42,2048,0,NN,yhat,1.1914704,0.035264004,1.1914182
48,4096,0,NN,yhat,1.1986816,-0.07134093,1.1947868
54,8192,0,NN,yhat,1.4593409,0.08069083,1.4542842
